# Data Preprocessing
## CVD Fairness Dissertation — NB0

**Purpose:** Load raw data, recode gender, convert age, remove clinically implausible records, and save the cleaned dataset.

**Input:** `data/raw/cardio_train.csv`  
**Output:** `data/processed/cardio_baseline_clean.csv`

**Does NOT contain:** EDA, distributions, correlations — those are in `02_eda.ipynb`

### Dataset Description

There are 3 types of input features:

*Objective*: factual information; *Examination*: results of medical examination; *Subjective*: information given by the patient.

| Feature | Type | Notes |
|---|---|---|
| age | Objective | int (days) |
| height | Objective | int (cm) |
| weight | Objective | float (kg) |
| gender | Objective | categorical code |
| ap_hi | Examination | Systolic blood pressure |
| ap_lo | Examination | Diastolic blood pressure |
| cholesterol | Examination | 1: normal, 2: above normal, 3: well above normal |
| gluc | Examination | 1: normal, 2: above normal, 3: well above normal |
| smoke | Subjective | binary |
| alco | Subjective | binary |
| active | Subjective | binary |
| cardio | Target | binary — presence/absence of CVD diagnosis |

**Limitation:** The target variable records diagnostic outcome rather than confirmed biological disease presence. Given well-documented systematic underdiagnosis of CVD in women, a negative label cannot be assumed to reliably confirm absence of disease in female patients.

## 1. Imports and Paths

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

RAW_DATA   = Path("../raw/cardio_train.csv")
OUTPUT_DIR = Path("../../outputs/preprocessing")
CLEAN_OUT  = Path("../processed/cardio_baseline_clean.csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Raw Data and Recode Gender

Gender is recoded from the original encoding (1=female, 2=male) to a clearer binary (0=female, 1=male) used consistently throughout the project.

In [15]:
data = pd.read_csv(RAW_DATA, sep=",")
data["gender"] = data["gender"].replace({1: 0, 2: 1})  # 0=female, 1=male

assert set(data["gender"].unique()) == {0, 1}, "Unexpected gender values after recode"
assert data.isnull().sum().sum() == 0, "Null values found in raw data"

print(f"Raw dataset shape : {data.shape}")
print(f"Gender values     : {sorted(data['gender'].unique())}  (0=female, 1=male)")
print(f"Female            : {(data['gender']==0).sum():,}  ({(data['gender']==0).mean()*100:.1f}%)")
print(f"Male              : {(data['gender']==1).sum():,}  ({(data['gender']==1).mean()*100:.1f}%)")

Raw dataset shape : (70000, 13)
Gender values     : [np.int64(0), np.int64(1)]  (0=female, 1=male)
Female            : 45,530  (65.0%)
Male              : 24,470  (35.0%)


## 3. Age Conversion and BMI

Age is converted from days to years. BMI is computed temporarily to help identify implausible height/weight combinations but is not retained in the final dataset.

In [16]:
data["age_years"] = data["age"] / 365.25
data["BMI"] = data["weight"] / (data["height"] / 100) ** 2

print("Age (years) summary:")
print(data["age_years"].describe().round(1))
print(f"\nBMI > 70 count : {(data['BMI'] > 70).sum()}")

Age (years) summary:
count    70000.0
mean        53.3
std          6.8
min         29.6
25%         48.4
50%         53.9
75%         58.4
max         64.9
Name: age_years, dtype: float64

BMI > 70 count : 36


## 4. Identify Implausible Values

Before applying filters, the extent of implausible values is checked to justify the exclusion thresholds used below.

In [17]:
print("Implausible blood pressure values in raw data:")
print(f"  Systolic > 250       : {(data['ap_hi'] > 250).sum()}  (range: {data[data['ap_hi']>250]['ap_hi'].min()}–{data[data['ap_hi']>250]['ap_hi'].max()})")
print(f"  Diastolic > 150      : {(data['ap_lo'] > 150).sum()}  (range: {data[data['ap_lo']>150]['ap_lo'].min()}–{data[data['ap_lo']>150]['ap_lo'].max()})")
print(f"  Systolic < 70        : {(data['ap_hi'] < 70).sum()}")
print(f"  Diastolic < 40       : {(data['ap_lo'] < 40).sum()}")
print(f"  Diastolic > Systolic : {(data['ap_lo'] > data['ap_hi']).sum()}")

Implausible blood pressure values in raw data:
  Systolic > 250       : 40  (range: 309–16020)
  Diastolic > 150      : 975  (range: 160–11000)
  Systolic < 70        : 189
  Diastolic < 40       : 59
  Diastolic > Systolic : 1234


## 5. Apply Filters

Records with clinically implausible values are removed. Each filter is logged showing how many records were kept and removed at each step.

In [18]:
def apply_filter(df, mask, name):
    before = len(df)
    df = df[mask].copy()
    after = len(df)
    print(f"{name:35s} | kept {after:6d}/{before:6d} | removed {before - after:6d}")
    return df

df = data.copy()

df = apply_filter(df, (df["age_years"] >= 30) & (df["age_years"] <= 65),  "Age 30–65 years")
df = apply_filter(df, (df["height"] >= 140)   & (df["height"] <= 220),    "Height 140–220 cm")
df = apply_filter(df, (df["weight"] >= 45)    & (df["weight"] <= 200),    "Weight 45–200 kg")
df = apply_filter(df, (df["ap_hi"] >= 70)     & (df["ap_hi"] <= 200),     "Systolic (ap_hi) 70–200")
df = apply_filter(df, (df["ap_lo"] >= 40)     & (df["ap_lo"] <= 150),     "Diastolic (ap_lo) 40–150")
df = apply_filter(df, df["ap_lo"] <= df["ap_hi"],                          "Diastolic <= Systolic")
df = apply_filter(df, df["cholesterol"].isin([1, 2, 3]),                   "Cholesterol in {1,2,3}")
df = apply_filter(df, df["gluc"].isin([1, 2, 3]),                          "Glucose in {1,2,3}")
for col in ["smoke", "alco", "active", "cardio"]:
    df = apply_filter(df, df[col].isin([0, 1]), f"{col} in {{0,1}}")

df = df.drop(columns=["id", "age", "BMI"])

print(f"\nFinal cleaned shape: {df.shape}")

Age 30–65 years                     | kept  69997/ 70000 | removed      3
Height 140–220 cm                   | kept  69844/ 69997 | removed    153
Weight 45–200 kg                    | kept  69551/ 69844 | removed    293
Systolic (ap_hi) 70–200             | kept  69266/ 69551 | removed    285
Diastolic (ap_lo) 40–150            | kept  68263/ 69266 | removed   1003
Diastolic <= Systolic               | kept  68177/ 68263 | removed     86
Cholesterol in {1,2,3}              | kept  68177/ 68177 | removed      0
Glucose in {1,2,3}                  | kept  68177/ 68177 | removed      0
smoke in {0,1}                      | kept  68177/ 68177 | removed      0
alco in {0,1}                       | kept  68177/ 68177 | removed      0
active in {0,1}                     | kept  68177/ 68177 | removed      0
cardio in {0,1}                     | kept  68177/ 68177 | removed      0

Final cleaned shape: (68177, 12)


## 6. Validate Cleaning Preserved Key Distributions

Checks that removing implausible records did not accidentally alter the sex ratio or class balance.

In [19]:
feature_meta = {
    "cardio":      ("Target",       "0:No CVD, 1:CVD"),
    "gender":      ("Gender (Sex)", "0=female, 1=male"),
    "cholesterol": ("Cholesterol",  "1:2:3"),
    "gluc":        ("Glucose",      "1:2:3"),
    "smoke":       ("Smoking",      "0:1"),
    "alco":        ("Alcohol",      "0:1"),
    "active":      ("Active",       "0:1"),
}

def compute_ratios(df):
    female = df[df["gender"] == 0]
    male   = df[df["gender"] == 1]
    def ratio_str(subset, col):
        cats   = sorted(subset[col].dropna().unique())
        counts = subset[col].value_counts(normalize=True).sort_index() * 100
        counts = counts.reindex(cats, fill_value=0)
        return ":".join([f"{v:.1f}" for v in counts])
    return {col: {"Total": ratio_str(df, col), "Female": ratio_str(female, col), "Male": ratio_str(male, col)}
            for col in feature_meta}

ratios_before = compute_ratios(data)
ratios_after  = compute_ratios(df)

print(f"{'Feature':<15} {'Before (Total)':<20} {'After (Total)':<20}")
print("-" * 55)
for col in feature_meta:
    print(f"{col:<15} {ratios_before[col]['Total']:<20} {ratios_after[col]['Total']:<20}")

# Key assertions
female_frac = (df["gender"] == 0).mean()
cvd_frac    = df["cardio"].mean()
assert 0.62 < female_frac < 0.68, f"Sex ratio shifted after cleaning: {female_frac:.3f}"
assert 0.48 < cvd_frac    < 0.52, f"Class balance shifted after cleaning: {cvd_frac:.3f}"
print("\nDistribution checks passed ✓")

Feature         Before (Total)       After (Total)       
-------------------------------------------------------
cardio          50.0:50.0            50.5:49.5           
gender          65.0:35.0            65.0:35.0           
cholesterol     74.8:13.6:11.5       75.0:13.5:11.5      
gluc            85.0:7.4:7.6         85.0:7.4:7.6        
smoke           91.2:8.8             91.2:8.8            
alco            94.6:5.4             94.7:5.3            
active          19.6:80.4            19.7:80.3           

Distribution checks passed ✓


## 7. Save Cleaned Dataset

In [20]:
df.to_csv(CLEAN_OUT, index=False)

print(f"Saved → {CLEAN_OUT}")
print(f"Final shape : {df.shape}")
print(f"Columns     : {list(df.columns)}")

Saved → ..\processed\cardio_baseline_clean.csv
Final shape : (68177, 12)
Columns     : ['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'age_years']


## 7. Distribution Table — Before and After Cleaning

In [21]:
table_rows = [
    [
        display_name,
        cat_label,
        ratios_before[col]["Total"],
        ratios_before[col]["Female"],
        ratios_before[col]["Male"],
        ratios_after[col]["Total"],
        ratios_after[col]["Female"],
        ratios_after[col]["Male"],
    ]
    for col, (display_name, cat_label) in feature_meta.items()
]

def render_table_image(rows, output_path):
    col_widths  = [0.12, 0.15, 0.08, 0.09, 0.08, 0.08, 0.09, 0.08]
    row_h       = 0.055
    left_start  = 0.02
    cell_pad    = 0.008
    header_y    = 0.97
    total_w = sum(col_widths)
    fig, ax = plt.subplots(figsize=(total_w * 18, len(rows) * 0.42 + 1.2))
    ax.axis("off")

    def draw_cell(ax, x, y, w, h, text, bold=False, fontsize=9, align='left', bg=None):
        if bg:
            ax.add_patch(mpatches.FancyBboxPatch(
                (x, y - h), w - 0.004, h,
                boxstyle="square,pad=0", linewidth=0.5,
                edgecolor='#aaaaaa', facecolor=bg,
                transform=ax.transAxes, clip_on=False
            ))
        tx = x + cell_pad if align == 'left' else x + w / 2
        ax.text(tx, y - h / 2, text, transform=ax.transAxes, fontsize=fontsize,
                fontweight='bold' if bold else 'normal',
                ha=align, va='center', color='#1a1a1a')

    x = left_start
    draw_cell(ax, x, header_y, col_widths[0], row_h * 2, "Feature",            bold=True, bg='#f0f0f0')
    x += col_widths[0]
    draw_cell(ax, x, header_y, col_widths[1], row_h * 2, "Categories/\nUnits", bold=True, bg='#f0f0f0')
    x += col_widths[1]
    before_w = sum(col_widths[2:5])
    after_w  = sum(col_widths[5:8])
    draw_cell(ax, x, header_y, before_w, row_h, "Distribution before", bold=True, align='center', bg='#dce8f5')
    x += before_w
    draw_cell(ax, x, header_y, after_w,  row_h, "Distribution after",  bold=True, align='center', bg='#d5ead5')

    sub_y = header_y - row_h
    x = left_start + col_widths[0] + col_widths[1]
    for i, lbl in enumerate(["Total", "Female", "Male"]):
        draw_cell(ax, x, sub_y, col_widths[2 + i], row_h, lbl, bold=True, fontsize=8, align='center', bg='#e8eef5')
        x += col_widths[2 + i]
    for i, lbl in enumerate(["Total", "Female", "Male"]):
        draw_cell(ax, x, sub_y, col_widths[5 + i], row_h, lbl, bold=True, fontsize=8, align='center', bg='#e2f0e2')
        x += col_widths[5 + i]

    for r_idx, row in enumerate(rows):
        y_top = header_y - row_h * 2 - r_idx * row_h
        bg    = '#ffffff' if r_idx % 2 == 0 else '#f9f9f9'
        x     = left_start
        for c_idx, val in enumerate(row):
            align = 'left' if c_idx < 2 else 'center'
            draw_cell(ax, x, y_top, col_widths[c_idx], row_h, str(val), fontsize=8, align=align, bg=bg)
            x += col_widths[c_idx]

    total_h = row_h * 2 + row_h * len(rows)
    ax.add_patch(mpatches.FancyBboxPatch(
        (left_start, header_y - total_h), total_w, total_h,
        boxstyle="square,pad=0", linewidth=1.2,
        edgecolor='#555555', facecolor='none',
        transform=ax.transAxes, clip_on=False
    ))

    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"Saved → {output_path}")

render_table_image(table_rows, OUTPUT_DIR / "distribution_table.png")

Saved → ..\..\outputs\preprocessing\distribution_table.png
